In [18]:
import os
import requests
import random
import time
from PIL import Image
from io import BytesIO

# --- KONFIGURACJA ---
SAVE_DIR = "lunar_surface_dataset"
TARGET_COUNT = 10

# Przełącznik pojedynczego trybu:
# DOWNLOAD_MODE = "preview"
# DOWNLOAD_MODE = "detail"
DOWNLOAD_MODE = "detail_plus"

# Szybki test obok siebie: podaj listę trybów albo [] żeby użyć tylko DOWNLOAD_MODE
QUICK_COMPARE_MODES = ["preview", "detail", "detail_plus"]

MODE_CONFIG = {
    "preview": {
        "image_size": 1024,
        "delta_deg": 2.0,
        "wms_format": "image/png",
        "file_ext": "png",
        "pil_format": "PNG",
        "sleep_seconds": 0.5,
    },
    "detail": {
        "image_size": 1536,
        "delta_deg": 0.80,
        "wms_format": "image/png",
        "file_ext": "png",
        "pil_format": "PNG",
        "sleep_seconds": 1.0,
    },
    "detail_plus": {
        "image_size": 2048,
        "delta_deg": 1.00,
        "wms_format": "image/png",
        "file_ext": "png",
        "pil_format": "PNG",
        "sleep_seconds": 1.2,
    },
}

if DOWNLOAD_MODE not in MODE_CONFIG:
    raise ValueError(f"Nieznany tryb: {DOWNLOAD_MODE}. Użyj 'preview', 'detail' albo 'detail_plus'.")

invalid_modes = [m for m in QUICK_COMPARE_MODES if m not in MODE_CONFIG]
if invalid_modes:
    raise ValueError(f"Nieznane tryby w QUICK_COMPARE_MODES: {invalid_modes}")

# Domyślny zestaw parametrów (dla pojedynczego trybu)
cfg = MODE_CONFIG[DOWNLOAD_MODE]
IMAGE_SIZE = cfg["image_size"]
DELTA_DEG = cfg["delta_deg"]
WMS_IMAGE_FORMAT = cfg["wms_format"]
FILE_EXT = cfg["file_ext"]
PIL_SAVE_FORMAT = cfg["pil_format"]
SLEEP_SECONDS = cfg["sleep_seconds"]

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# Endpoint serwera WMS USGS
WMS_URL = "https://planetarymaps.usgs.gov/cgi-bin/mapserv"

# Musi być "earth", serwery USGS tak katalogują mapę Księżyca
WMS_MAP = "/maps/earth/moon_simp_cyl.map"
LAYER = "LROC_WAC" # Warstwa z Lunar Reconnaissance Orbiter

In [19]:
active_modes = QUICK_COMPARE_MODES if QUICK_COMPARE_MODES else [DOWNLOAD_MODE]

print(f"Rozpoczynam pobieranie {TARGET_COUNT} losowych wycinków satelitarnych Księżyca...")
print(f"Aktywne tryby: {', '.join(active_modes)}")

for mode_name in active_modes:
    mode_cfg = MODE_CONFIG[mode_name]
    image_size = mode_cfg["image_size"]
    delta = mode_cfg["delta_deg"]
    wms_image_format = mode_cfg["wms_format"]
    file_ext = mode_cfg["file_ext"]
    pil_save_format = mode_cfg["pil_format"]
    sleep_seconds = mode_cfg["sleep_seconds"]

    mode_save_dir = os.path.join(SAVE_DIR, mode_name) if len(active_modes) > 1 else SAVE_DIR
    os.makedirs(mode_save_dir, exist_ok=True)

    print(f"\n--- Tryb: {mode_name} | size={image_size}px | delta={delta}deg | format={file_ext} ---")

    for i in range(TARGET_COUNT):
        # Losowanie współrzędnych
        lon_min = random.uniform(-170, 160)
        lat_min = random.uniform(-60, 60)

        lon_max = lon_min + delta
        lat_max = lat_min + delta

        # Kompletne parametry zapytania WMS 1.1.1
        params = {
            "map": WMS_MAP,
            "request": "GetMap",
            "service": "WMS",
            "version": "1.1.1",
            "layers": LAYER,
            "styles": "",  # Wymagany przez protokół WMS
            "srs": "EPSG:4326",
            "bbox": f"{lon_min},{lat_min},{lon_max},{lat_max}",
            "width": image_size,
            "height": image_size,
            "format": wms_image_format
        }

        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }

        try:
            response = requests.get(WMS_URL, params=params, headers=headers, timeout=20)
            response.raise_for_status()

            # Sprawdzenie, czy serwer zwrócił obraz
            if 'image' in response.headers.get('Content-Type', ''):
                img = Image.open(BytesIO(response.content))

                # Wymuszenie odcieni szarości
                img = img.convert("L")

                lon_tag = f"{lon_min:.3f}"
                lat_tag = f"{lat_min:.3f}"
                file_path = os.path.join(
                    mode_save_dir,
                    f"lroc_tile_{i+1:03d}_{mode_name}_Lo{lon_tag}_La{lat_tag}.{file_ext}"
                )
                img.save(file_path, pil_save_format)

                print(f"[{mode_name} {i+1}/{TARGET_COUNT}] Zapisano: (Lon: {lon_min:.3f}, Lat: {lat_min:.3f})")
            else:
                error_snippet = response.text[:250].replace('\n', ' ')
                print(f"[{mode_name} {i+1}/{TARGET_COUNT}] Błąd WMS! Odpowiedź serwera: {error_snippet}")

        except Exception as e:
            print(f"[{mode_name} {i+1}/{TARGET_COUNT}] Błąd sieci: {e}")

        time.sleep(sleep_seconds)

print(f"\nGotowe! Dataset został zapisany w folderze '{SAVE_DIR}'.")

Rozpoczynam pobieranie 10 losowych wycinków satelitarnych Księżyca...
Aktywne tryby: preview, detail, detail_plus

--- Tryb: preview | size=1024px | delta=2.0deg | format=png ---
[preview 1/10] Zapisano: (Lon: 18.964, Lat: -40.738)
[preview 2/10] Zapisano: (Lon: -140.992, Lat: -8.936)
[preview 3/10] Zapisano: (Lon: 76.923, Lat: -59.717)
[preview 4/10] Zapisano: (Lon: 8.202, Lat: -40.400)
[preview 5/10] Zapisano: (Lon: 55.649, Lat: -46.591)
[preview 6/10] Zapisano: (Lon: 25.086, Lat: 59.930)
[preview 7/10] Zapisano: (Lon: -99.705, Lat: 59.776)
[preview 8/10] Zapisano: (Lon: -87.435, Lat: -20.767)
[preview 9/10] Zapisano: (Lon: 83.077, Lat: 42.666)
[preview 10/10] Zapisano: (Lon: 151.577, Lat: -39.471)

--- Tryb: detail | size=1536px | delta=0.8deg | format=png ---
[detail 1/10] Zapisano: (Lon: 105.230, Lat: -16.180)
[detail 2/10] Zapisano: (Lon: 21.355, Lat: 16.279)
[detail 3/10] Zapisano: (Lon: -49.808, Lat: 40.781)
[detail 4/10] Zapisano: (Lon: -15.467, Lat: 5.931)
[detail 5/10] Zapis